# Pre-Processing

In [1]:
# : Install libraries
!pip install transformers datasets scikit-learn pandas numpy

In [2]:
#: Import essentials
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import BertTokenizerFast

In [3]:
#: Load your dataset (must have columns: 'text' and 'label')
df = pd.read_csv("/content/Roman Urdu Tagged Dataset.csv")  # Replace with your file
print(df.head())

                                                Text Sentiment (POS/NEG/NEU)
0  Shan Food ki quality bohat zabardast ha ...boh...                Positive
1                               ye bohat mazaydar ha                Positive
2  Shan food bohat achi company hain, mujay in k ...                Positive
3   bohat acha pakistani brand ha..zabardast quality                Positive
4  Hamare ghar me yehi msale use hote hain meri a...                Positive


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21940 entries, 0 to 21939
Data columns (total 2 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Text                     21940 non-null  object
 1   Sentiment (POS/NEG/NEU)  21940 non-null  object
dtypes: object(2)
memory usage: 342.9+ KB


In [4]:
df['Sentiment (POS/NEG/NEU)'].value_counts()

,count
Sentiment (POS/NEG/NEU),
Neutral,11560
Positive,5634
Negative,4746


In [5]:
# balancing dataset
from sklearn.utils import resample
import pandas as pd

# Separate classes
df_neu = df[df['Sentiment (POS/NEG/NEU)'] == 'Neutral']
df_pos = df[df['Sentiment (POS/NEG/NEU)'] == 'Positive']
df_neg = df[df['Sentiment (POS/NEG/NEU)'] == 'Negative']

# Find majority class size
max_size = max(len(df_neu), len(df_pos), len(df_neg))

# Oversample minority classes
df_pos_upsampled = resample(df_pos, replace=True, n_samples=max_size, random_state=42)
df_neg_upsampled = resample(df_neg, replace=True, n_samples=max_size, random_state=42)
df_neu_upsampled = resample(df_neu, replace=True, n_samples=max_size, random_state=42)

# Combine and shuffle
df_balanced = pd.concat([df_neu_upsampled, df_pos_upsampled, df_neg_upsampled])
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(df_balanced['Sentiment (POS/NEG/NEU)'].value_counts())


Sentiment (POS/NEG/NEU)
Neutral     11560
Positive    11560
Negative    11560
Name: count, dtype: int64


In [ ]:
df.head()

,Text,Sentiment (POS/NEG/NEU)
0,Shan Food ki quality bohat zabardast ha ...boh...,Positive
1,ye bohat mazaydar ha,Positive
2,"Shan food bohat achi company hain, mujay in k ...",Positive
3,bohat acha pakistani brand ha..zabardast quality,Positive
4,Hamare ghar me yehi msale use hote hain meri a...,Positive


In [6]:
from sklearn.model_selection import train_test_split
import pandas as pd

# Rename column
df_balanced= df_balanced.rename(columns={"Sentiment (POS/NEG/NEU)": "sentiment"})

# Map labels to integers
label_map = {'Positive': 0, 'Negative': 1, 'Neutral': 2}
df_balanced['label'] = df_balanced['sentiment'].map(label_map)

# Shuffle before splitting
df = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)



In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34680 entries, 0 to 34679
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Text       34680 non-null  object
 1   sentiment  34680 non-null  object
 2   label      34680 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 812.9+ KB


In [7]:
df = df[['Text', 'label']].rename(columns={'Text': 'text'})
print(df.head())

                                                text  label
0                           girlfriend set krli kia?      2
1  Careem walo. Iski shirt utarwa k pics banwao t...      1
2  jab naam maha hai tou dekhne mai bhi khubsurat...      0
3                                 dikhna ma chi hay       0
4   is lye like kya tha k weather k bare m update...      2


In [ ]:
df.head()

,text,label
0,girlfriend set krli kia?,2
1,Careem walo. Iski shirt utarwa k pics banwao t...,1
2,jab naam maha hai tou dekhne mai bhi khubsurat...,0
3,dikhna ma chi hay,0
4,is lye like kya tha k weather k bare m update...,2


In [8]:
# Split into train (80%), validation (10%), test (10%)
train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42)

# Print sizes
print("✅ Train:", len(train_df), " Validation:", len(val_df), " Test:", len(test_df))
print(train_df['label'].value_counts())

✅ Train: 27744  Validation: 3468  Test: 3468
label
1    9248
0    9248
2    9248
Name: count, dtype: int64


In [9]:
# Step: Tokenization using Hugging Face Datasets + Tokenizer
# Install first if needed: !pip install transformers datasets

from datasets import Dataset
from transformers import AutoTokenizer

In [10]:
#  Use multilingual BERT (best for Urdu/Roman Urdu + English)
MODEL_NAME = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

In [11]:
#  Convert pandas DataFrames to Hugging Face Datasets
train_ds = Dataset.from_pandas(train_df)
val_ds   = Dataset.from_pandas(val_df)
test_ds  = Dataset.from_pandas(test_df)

In [12]:
# Tokenize function
MAX_LEN = 128  # adjust if your sentences are long
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",   # ensures uniform length
        truncation=True,
        max_length=MAX_LEN
    )

In [13]:
# Apply tokenization
train_ds = train_ds.map(tokenize_function, batched=True)
val_ds   = val_ds.map(tokenize_function, batched=True)
test_ds  = test_ds.map(tokenize_function, batched=True)

Map:   0%|          | 0/27744 [00:00<?, ? examples/s]

Map:   0%|          | 0/3468 [00:00<?, ? examples/s]

Map:   0%|          | 0/3468 [00:00<?, ? examples/s]

In [14]:
#  Set format for PyTorch
train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

print("✅ Tokenization done.")
print(train_ds[0])

✅ Tokenization done.
{'label': tensor(1), 'input_ids': tensor([   101,  15128,  38806,  10113,  10126,  10262,  12715,  33478,  10116,
         10414,  10710,  45505,  13080,  10248,  20506,  55788,  10911,  15797,
         10525, 106629,    179,  10380,  61497,  10113,  10248,  32493,  13080,
           179,  18387,  40154,  10238,  10248,    102,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,  

Why this setup improves accuracy:

Uses multilingual BERT (handles Urdu + Roman Urdu well).

Uses max length padding for consistent batches.

Keeps truncation safe (prevents cutting mid-sentence).

Creates ready-to-use PyTorch tensors for the Trainer API.

Would you like me to give the next step: fine-tuning BERT with Hugging Face Trainer (accuracy-optimized)?

# Full Fine Tuning


In [ ]:
# Install first if needed: !pip install transformers evaluate accelerate
!pip install evaluate

from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate
import torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00


In [ ]:
# Load model (3 sentiment labels)
MODEL_NAME = "bert-base-multilingual-cased"
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Metric (accuracy)
metric = evaluate.load("accuracy")

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=labels)

In [ ]:
# Training arguments (tuned for accuracy + low data)
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",     # evaluate each epoch
    save_strategy="epoch",           # save best each epoch
    load_best_model_at_end=True,     # restore best model
    metric_for_best_model="accuracy",
    greater_is_better=True,
    num_train_epochs=12,              # small dataset → fewer epochs
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,              # standard for BERT fine-tuning
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    seed=42,
    fp16=torch.cuda.is_available(),  # automatic mixed precision for speed
    report_to="none"                 # disable wandb/tensorboard
)

In [ ]:
# Trainer setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

/tmp/ipython-input-4205485688.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
# Train
trainer.train()
# Evaluate
# ---------------------------
results = trainer.evaluate()
print("Final Accuracy:", results["eval_accuracy"])

Epoch,Training Loss,Validation Loss,Accuracy
1,0.630500,0.567245,0.774510
2,0.476700,0.390475,0.861592
3,0.269300,0.341547,0.897059
4,0.225800,0.326237,0.914937
5,0.126700,0.423587,0.926182
6,0.144000,0.379409,0.927047
7,0.069800,0.422472,0.934833
8,0.051600,0.411795,0.937140
9,0.031500,0.445057,0.938293
10,0.007100,0.468113,0.943483


Final Accuracy: 0.9440599769319492


In [ ]:
trainer.predict(test_ds)


PredictionOutput(predictions=array([[-3.140625 , -3.6875   ,  6.9296875],
       [-3.0820312, -3.7636719,  6.9335938],
       [-3.1152344, -3.7207031,  6.9335938],
       ...,
       [ 6.9179688, -4.6328125, -2.6347656],
       [-3.3007812, -3.5117188,  6.9335938],
       [ 6.9453125, -4.6601562, -2.6796875]], dtype=float32), label_ids=array([2, 2, 2, ..., 0, 2, 0]), metrics={'test_loss': 0.4926615357398987, 'test_accuracy': 0.9437716262975778, 'test_runtime': 6.7639, 'test_samples_per_second': 512.721, 'test_steps_per_second': 32.082})

# LORA

In [ ]:
# ✅ Step 1: Install required libraries
!pip install -q transformers peft accelerate evaluate bitsandbytes datasets
!pip install --upgrade transformers

In [ ]:
import torch
from transformers import BertForSequenceClassification, BertTokenizerFast, TrainingArguments, Trainer, BitsAndBytesConfig
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
import evaluate

# ---------------------------
# 1. Load tokenizer and model
# ---------------------------
# Re-initialize the model to ensure it's a fresh, un-PEFT-modified base model
model_name = "bert-base-multilingual-cased"
tokenizer = BertTokenizerFast.from_pretrained(model_name)

model = BertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,           # <-- change for your dataset
    device_map="auto"
)

# 2. Prepare LoRA configuration
# ---------------------------

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS",
    target_modules=["query", "key", "value", "output.dense"]  # strongest effect
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ---------------------------
# 3. Load dataset
# ---------------------------
# The dataset loading and preparation has been done in previous cells.
# Use the already prepared datasets from previous steps.
train_set = train_ds
test_set = test_ds

# ---------------------------
# 4. Tokenization
# ---------------------------
# Tokenization is already handled in previous cells and train_ds, test_ds are ready.

# ---------------------------
# 5. Metric — Accuracy
# ---------------------------
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return accuracy.compute(predictions=preds, references=labels)

# ---------------------------
# 6. Training Arguments

training_args = TrainingArguments(
    output_dir="./mb_lora",

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,

    learning_rate=2e-4,               # LoRA needs higher LR
    num_train_epochs=15,

    fp16=True,

    eval_strategy="epoch", # Changed from evaluation_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,

    weight_decay=0.01,                # ★ improves generalization
    warmup_ratio=0.1,                 # ★ stabilizes training
    lr_scheduler_type="cosine",       # ★ smoother LR → better accuracy

    logging_steps=50,
    seed=42,

    report_to="wandb",                # optional: enable W&B logging
    run_name="bert_lora_optimized"
)

# ---------------------------
# 7. Trainer
# ---------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_set,
    eval_dataset=test_set,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# ---------------------------
# 8. Train
# ---------------------------
trainer.train()

# ---------------------------
# 9. Evaluate
# ---------------------------
results = trainer.evaluate()
print("Final Accuracy:", results["eval_accuracy"])

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 1,919,235 || all params: 179,774,982 || trainable%: 1.0676


/tmp/ipython-input-3607128426.py:91: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The model is already on multiple devices. Skipping the move to device specified in `args`.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: WARNING Invalid choice
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: adeelahmed4868 (adeelahmed4868-riphah-international-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Accuracy
1,0.726400,0.666057,0.730969
2,0.596600,0.564400,0.774510
3,0.448000,0.440392,0.834487
4,0.444600,0.373184,0.861880
5,0.358400,0.362312,0.883795
6,0.331200,0.323582,0.893887
7,0.245700,0.339694,0.903691
8,0.217100,0.317210,0.911476
9,0.218500,0.299197,0.923875
10,0.110600,0.328661,0.925606


Final Accuracy: 0.9238754325259516


# Prefix Tuning (P-Tuning v2) for BERT

In [ ]:
!pip install -q transformers peft accelerate evaluate bitsandbytes datasets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 15.3 MB/s eta 0:00:00


In [ ]:
#  Configure Prefix Tuning
from peft import PrefixTuningConfig, get_peft_model, TaskType

# Optimized Prefix Tuning configuration
prefix_config = PrefixTuningConfig(
    task_type=TaskType.SEQ_CLS,  # sequence classification
    num_virtual_tokens=40,       # number of prefix tokens
    prefix_projection=True,      # improves representation learning
    inference_mode=False,
    encoder_hidden_size=768      # explicitly set the hidden size for projection
)

In [ ]:
from transformers import AutoModelForSequenceClassification

# Load a fresh base model for Prefix Tuning with the correct number of labels
base_model_for_prefix_tuning = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

# Apply Prefix Tuning to the fresh base model
model = get_peft_model(base_model_for_prefix_tuning, prefix_config)

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Show trainable parameters
model.print_trainable_parameters()

trainable params: 14,797,827 || all params: 192,653,574 || trainable%: 7.6811


Why these settings?

num_virtual_tokens=30 → good balance for small datasets (more tokens = more expressive power).

prefix_projection=True → adds a projection layer for better adaptation.

Only ~1–2% parameters trainable → efficient + stable.

In [ ]:
from transformers import TrainingArguments
import torch

training_args = TrainingArguments(
    output_dir="./results_prefix",

    # ------ Evaluation / Saving ------
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=False,     # SAFE now (fix shown below)
    metric_for_best_model="accuracy",
    greater_is_better=True,

    # ------ Hyperparameters ------
    learning_rate=3e-4,              # good for prefix tuning
    num_train_epochs=15,              # prefix tuning benefits from more epochs
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_ratio=0.1,
    weight_decay=0.01,

    # ------ Precision ------
    fp16=torch.cuda.is_available(),

    # ------ Logging / WandB ------
    logging_steps=50,
    report_to="wandb",               # ENABLE WandB
    run_name="prefix_tuning_experiment",  # WandB experiment name

    # ------ Reproducibility ------
    seed=42
)


In [ ]:
from transformers import Trainer
import evaluate
import numpy as np

# Load accuracy metric
accuracy = evaluate.load("accuracy")

# Metric function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return accuracy.compute(predictions=preds, references=labels)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

/tmp/ipython-input-2929103847.py:14: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
# 🔥 Train Prefix Tuning model
trainer.train()
#  Evaluate on test set
# ---------------------------
results = trainer.evaluate(test_set)
print("Final Test Accuracy:", results["eval_accuracy"])

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: adeelahmed4868 (adeelahmed4868-riphah-international-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Accuracy
1,1.119900,1.108280,0.333333
2,1.100900,1.099187,0.338524
3,1.099700,1.101257,0.333333
4,1.101100,1.102096,0.333333
5,1.101200,1.099281,0.333333
6,1.102000,1.100911,0.333333
7,1.098500,1.098630,0.333333
8,1.097700,1.098840,0.333333
9,1.100000,1.098634,0.333333
10,1.099600,1.099211,0.333333


KeyboardInterrupt: 

In [ ]:
# Evaluate
results = trainer.evaluate(test_ds)
print("✅ Prefix Tuning Test Results:", results)

Epoch,Training Loss,Validation Loss,Accuracy
1,1.119900,1.108280,0.333333
2,1.100900,1.099187,0.338524
3,1.099700,1.101257,0.333333
4,1.101100,1.102096,0.333333
5,1.101200,1.099281,0.333333
6,1.102000,1.100911,0.333333
7,1.098500,1.098630,0.333333
8,1.097700,1.098840,0.333333
9,1.100000,1.098634,0.333333
10,1.099600,1.099211,0.333333


✅ Prefix Tuning Test Results: {'eval_loss': 1.098647117614746, 'eval_accuracy': 0.3333333333333333}


Why Prefix Tuning might outperform LoRA sometimes

Learns prefix vectors per layer → stronger contextual control.

Works very well on text generation & classification tasks.

For your multilingual sentiment data, it can match or even surpass LoRA if tuned slightly (try num_virtual_tokens=40).

If Prefix-Tuning (Prefix PEFT) is giving only ~33% accuracy, while full fine-tuning and LoRA gave >80%, that’s completely expected and practical in many real cases — especially on small or domain-specific datasets.

Let’s break this down clearly and practically 👇

⚙️ Why Prefix-Tuning accuracy is low

Prefix-tuning trains very few parameters

It doesn’t modify model weights, only adds “prefix vectors” to attention layers.

This is powerful for large-scale tasks with huge data, but not for small custom datasets.

It relies heavily on pretrained model knowledge

If your dataset (e.g. Urdu / literary English) is linguistically different from the pretrained model’s training data, prefix-tuning struggles to adapt.

LoRA or full fine-tuning change weights more directly → better adaptation.

Prefix length too short or learning rate too low

Common mistake: prefix length (num_virtual_tokens) too small (like 10 or 20).

Try increasing to 30–50.

Also tune learning rate: start from 2e-4 to 1e-3 for prefix-tuning.

Small dataset size

Prefix-tuning generally needs larger datasets to find useful prefix vectors.

# prompt tunning

In [ ]:
# Step 1: Load base model and tokenizer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PromptTuningConfig, get_peft_model

model_name = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Step 1b: Define prompt tuning configuration
prompt_config = PromptTuningConfig(
    task_type="SEQ_CLS",
    num_virtual_tokens=40,         # try 20–50 for best accuracy
    tokenizer_name_or_path=model_name,
)

In [ ]:
# Step 1c: Create PEFT model
peft_model = get_peft_model(model, prompt_config)
peft_model.print_trainable_parameters()

trainable params: 33,027 || all params: 177,888,774 || trainable%: 0.0186


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./prompt_tuned_bert",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-4,            # 🔥 higher LR helps prompt tuning
    num_train_epochs=15,            # try 5–8 for small data
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_dir="./logs",
    logging_steps=50,
    save_total_limit=2,
    report_to="none"  # Ensure W&B is disabled
)

In [ ]:
import numpy as np
import evaluate

# Load accuracy metric
accuracy = evaluate.load("accuracy")

# Metric function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return accuracy.compute(predictions=preds, references=labels)

# Create Trainer
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)


/tmp/ipython-input-431711176.py:14: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"


In [ ]:
# Train the model
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,1.063800,1.045938,0.454441
2,1.034100,1.001029,0.498270
3,1.005300,0.965594,0.531719
4,0.984300,0.945825,0.543829
5,1.005500,0.934821,0.555652
6,0.956000,0.918030,0.562860
7,0.941600,0.909639,0.562572
8,0.927000,0.896380,0.576701
9,0.941700,0.891909,0.585640
10,0.941000,0.880740,0.589965


Epoch,Training Loss,Validation Loss,Accuracy
1,1.063800,1.045938,0.454441
2,1.034100,1.001029,0.498270
3,1.005300,0.965594,0.531719
4,0.984300,0.945825,0.543829
5,1.005500,0.934821,0.555652
6,0.956000,0.918030,0.562860
7,0.941600,0.909639,0.562572
8,0.927000,0.896380,0.576701
9,0.941700,0.891909,0.585640
10,0.941000,0.880740,0.589965


TrainOutput(global_step=26010, training_loss=0.9782638417074929, metrics={'train_runtime': 2419.6624, 'train_samples_per_second': 171.991, 'train_steps_per_second': 10.749, 'total_flos': 2.737505732272128e+16, 'train_loss': 0.9782638417074929, 'epoch': 15.0})

In [ ]:
# Evaluate on test set
test_results = trainer.evaluate(test_ds)
print("✅ Test Results:", test_results)

✅ Test Results: {'eval_loss': 0.8480169773101807, 'eval_accuracy': 0.6208189158016147, 'eval_runtime': 7.543, 'eval_samples_per_second': 459.764, 'eval_steps_per_second': 14.45, 'epoch': 15.0}


# P-Tuning v2

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import PromptEncoderConfig, get_peft_model

# Load base model & tokenizer
model_name ="bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

# Define P-Tuning v2 configuration
peft_config = PromptEncoderConfig(
    task_type="SEQ_CLS",
    num_virtual_tokens=40,     # can tune between 20–50
    encoder_hidden_size=768,   # same as BERT hidden size
)

# Apply PEFT configuration
peft_model = get_peft_model(model, peft_config)
peft_model.print_trainable_parameters()


model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 1,804,803 || all params: 179,660,550 || trainable%: 1.0046


In [ ]:
pip install evaluate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.5 MB/s eta 0:00:00


In [ ]:
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np

# ✅ Training arguments (optimized for P-Tuning v2)
training_args = TrainingArguments(
    output_dir="./p_tuning_v2_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-4,         # slightly higher LR for soft prompt learning
    num_train_epochs=15,         # 5–8 works best
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    save_total_limit=2,
    logging_dir="./logs",
    logging_steps=50,
    report_to="none"  # Disable W&B
)

# ✅ Metric
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return accuracy.compute(predictions=preds, references=labels)

# ✅ Trainer
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

/tmp/ipython-input-3951469594.py:32: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
# ✅: Train and Evaluate P-Tuning v2 model

# Train
trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy
1,1.016500,0.982148,0.478085
2,0.964700,0.932026,0.523933
3,0.891900,0.934501,0.556228
4,0.890100,0.898636,0.569204
5,0.864800,0.907656,0.557093
6,0.835500,0.883546,0.597751
7,0.801800,0.855594,0.613899
8,0.820000,0.840046,0.627163
9,0.821400,0.831081,0.628893
10,0.793300,0.879110,0.619954


TrainOutput(global_step=26010, training_loss=0.8662625352037453, metrics={'train_runtime': 8184.7005, 'train_samples_per_second': 50.846, 'train_steps_per_second': 3.178, 'total_flos': 2.737505732272128e+16, 'train_loss': 0.8662625352037453, 'epoch': 15.0})

In [ ]:
# Evaluate on test set
test_results = trainer.evaluate(test_ds)
print("✅ Test Results:", test_results)

✅ Test Results: {'eval_loss': 1.1308259963989258, 'eval_accuracy': 0.4596309111880046, 'eval_runtime': 29.0794, 'eval_samples_per_second': 119.26, 'eval_steps_per_second': 3.748, 'epoch': 15.0}


In [ ]:
 # Evaluate P-Tuning v2 model

# Evaluate on validation set
val_results = trainer.evaluate()
print("✅ Validation Results:", val_results)

# Evaluate on test set
test_results = trainer.evaluate(test_ds)
print("✅ Test Results:", test_results)


✅ Validation Results: {'eval_loss': 1.150259017944336, 'eval_accuracy': 0.4524221453287197, 'eval_runtime': 29.9682, 'eval_samples_per_second': 115.723, 'eval_steps_per_second': 3.637, 'epoch': 15.0}
✅ Test Results: {'eval_loss': 1.1308259963989258, 'eval_accuracy': 0.4596309111880046, 'eval_runtime': 31.4546, 'eval_samples_per_second': 110.254, 'eval_steps_per_second': 3.465, 'epoch': 15.0}


AdaLoRA

In [15]:
# Run in a notebook cell / terminal
!pip install -q transformers peft accelerate evaluate bitsandbytes datasets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 14.8 MB/s eta 0:00:00


In [16]:
import os, random, numpy as np, torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


In [17]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "bert-base-multilingual-cased"   # best for Urdu + Roman Urdu + English
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)


model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [18]:
from peft import AdaLoraConfig, get_peft_model, TaskType


adalora_config = AdaLoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,                  # start rank (slightly larger for better accuracy)
    lora_alpha=32,
    target_modules=["query", "value"],  # adapt attention Q/V (good default for BERT)
    lora_dropout=0.05,
    inference_mode=False,
    init_r=16,             # initial rank
    target_r=6,            # final target low-rank
    tinit=100,             # iterations before adaptation starts
    tfinal=600,            # when adaptation finishes (adjust if dataset small)
    deltaT=10,              # adaptation frequency
    total_step=1000 # Placeholder value, should be calculated based on dataset size, batch size and epochs
)

adalora_model = get_peft_model(base_model, adalora_config)
adalora_model.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/peft/tuners/adalora/config.py:96: UserWarning: Note that `r` is not used in AdaLora and will be ignored.If you intended to set the initial rank, use `init_r` instead.
  warnings.warn(


trainable params: 592,515 || all params: 178,448,286 || trainable%: 0.3320


Notes: r=12 gives stronger capacity than r=8 — helps accuracy on domain data. target_modules=["query","value"] focuses adaptation where it matters.

In [19]:
import evaluate, numpy as np
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=labels)


Step 8 — Trainer initialization (include early stopping callback)

In [20]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./adalora_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    learning_rate=3e-4,            # good for adapter methods
    num_train_epochs=15,            # small dataset: 4-8
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    seed=SEED,
    dataloader_pin_memory=False,   # set False to avoid 'pin_memory' warning if on CPU
    logging_steps=50,
    save_total_limit=2,
    report_to="none"               # disable automatic logging services
)

In [21]:
from transformers import Trainer
from transformers import EarlyStoppingCallback

trainer = Trainer(
    model=adalora_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]  # stops if no val improvement for 2 epochs
)


/tmp/ipython-input-3692918051.py:4: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Why early stopping: prevents overfitting and selects best epoch automatically.

 — Train (monitoring best model automatically)

In [22]:
# Disable wandb if accidentally enabled
import os
os.environ["WANDB_DISABLED"] = "true"

trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy
1,1.038400,1.006202,0.487889
2,0.837200,0.769804,0.663783
3,0.674600,0.687618,0.705882
4,0.710000,0.619035,0.749423
5,0.612700,0.578569,0.765859
6,0.603200,0.534647,0.783449
7,0.516100,0.512356,0.797001
8,0.429000,0.496920,0.810842
9,0.477200,0.469001,0.832468
10,0.405100,0.435095,0.842561


Epoch,Training Loss,Validation Loss,Accuracy
1,1.038400,1.006202,0.487889
2,0.837200,0.769804,0.663783
3,0.674600,0.687618,0.705882
4,0.710000,0.619035,0.749423
5,0.612700,0.578569,0.765859
6,0.603200,0.534647,0.783449
7,0.516100,0.512356,0.797001
8,0.429000,0.496920,0.810842
9,0.477200,0.469001,0.832468
10,0.405100,0.435095,0.842561


TrainOutput(global_step=26010, training_loss=0.5842933945177335, metrics={'train_runtime': 2698.4057, 'train_samples_per_second': 154.224, 'train_steps_per_second': 9.639, 'total_flos': 2.756370189164544e+16, 'train_loss': 0.5842933945177335, 'epoch': 15.0})

In [23]:
# ---------------------------
results = trainer.evaluate()
print("Final Accuracy:", results["eval_accuracy"])

Final Accuracy: 0.8711072664359861


Extra accuracy tips (do not skip)

Seed + determinism — we set seeds at Step 2.

Balanced data & stratified split — you already did.

Appropriate tokenizer/model — multilingual BERT chosen for Urdu.

Use load_best_model_at_end=True + early stopping — avoids overfitting.

Tune r and init_r — increase r (12→16) if GPU allows to boost accuracy.

Increase tfinal / lower tinit when dataset is large so adaptation has time to allocate ranks.

Monitor val accuracy each epoch — if it plateaus, increase epochs slightly or reduce LR.

Use FP16 on GPU — stable and faster.

Try larger MAX_LEN (e.g., 256) only if sentences are long.

If AdaLoRA underperforms LoRA/full-finetune, try increasing r or adapt target_modules (include "dense" or other layers).

If you want, I can now:

Run hyperparameter suggestions to try (list of 3 tuned configs), or

Provide a minimal script that logs epoch-by-epoch accuracies to a CSV, or

Give a one-line change to increase model capacity (e.g., r=16, init_r=16, target_r=6) and show where to change it.

Which of these would help you next?

# IA³ fine-tuning

Apply IA³ Configuration (efficiency + accuracy focused)

In [24]:
from peft import IA3Config, get_peft_model, TaskType

ia3_config = IA3Config(
    task_type=TaskType.SEQ_CLS,
    target_modules=["query", "value"],  # focus on attention parts for best results
    # feedforward_modules=["intermediate.dense"],  # optional for better adaptation
)

ia3_model = get_peft_model(base_model, ia3_config)
ia3_model.print_trainable_parameters()

trainable params: 20,739 || all params: 178,466,718 || trainable%: 0.0116


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:282: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [25]:
import evaluate, numpy as np
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=labels)


Step 7 — TrainingArguments (optimized for IA³)

Lower learning rate and small dropout give better convergence for lightweight PEFTs.

In [27]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./ia3_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    learning_rate=2e-4,              # slightly smaller than LoRA for stability
    num_train_epochs=15,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    save_total_limit=2,
    dataloader_pin_memory=False,
    seed=42,
    report_to="none"                 # turn off W&B etc.
)

Step 8 — Initialize Trainer with Early Stopping

In [28]:
from transformers import Trainer, EarlyStoppingCallback

trainer = Trainer(
    model=ia3_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)


/tmp/ipython-input-656194628.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [29]:
import os
os.environ["WANDB_DISABLED"] = "true"   # make sure W&B is off

trainer.train()

# ---------------------------
results = trainer.evaluate()
print("Final Accuracy:", results["eval_accuracy"])

Epoch,Training Loss,Validation Loss,Accuracy
1,0.349400,0.372913,0.871684
2,0.312400,0.375975,0.870819
3,0.254000,0.375764,0.870819


Final Accuracy: 0.871683967704729
